# Imports

In [5]:
import os

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

%matplotlib inline

from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler, MaxAbsScaler


/home/tiggi/Documents/IU_projects/bitcoin_rec/cc_fraud_NMF/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data loading

In [6]:
datafolder = './data'
data = os.path.join(datafolder, "1", "elliptic_bitcoin_dataset","elliptic_txs_features.csv")

datafolder = os.path.join(datafolder, "1","elliptic_bitcoin_dataset")   


There are 166 features to the dataset, 92 local and 74 aggregated. [add source - demistifying... ]
The CSV does not have the columns defined
txid - transaction id
time_step - 1-49 timestamps groups
lf_{number} - local feature number 1-92
af_{number} - aggregated feature number 1-74


In [7]:
cols = ['txId','time_step']


local_cols = [f'lf_{i+1}' for i in range(93)]
agg_cols = [f'af_{i+1}' for i in range(72)]

cols += local_cols + agg_cols

In [8]:
features = pd.read_csv(os.path.join(datafolder, "elliptic_txs_features.csv"),  index_col=False, names=cols,)
edges = pd.read_csv(os.path.join(datafolder, "elliptic_txs_edgelist.csv"))
classes = pd.read_csv(os.path.join(datafolder, "elliptic_txs_classes.csv"))


In [9]:
# map classes: licit - 0; illicit- 1;  unknown -2
class_mapping = {'2': 0, '1': 1, 'unknown': 2}
classes.replace({"class": class_mapping}, inplace=True)


,txId,class
0,230425980,2
1,5530458,2
2,232022460,2
3,232438397,0
4,230460314,2
...,...,...
203764,173077460,2
203765,158577750,2
203766,158375402,1
203767,158654197,2


In [10]:
df = pd.merge(features, classes, on='txId')

In [11]:
# feature enrichment - add edges
def add_degree_feature(target_df, input_edges):
        node_ids = set(target_df['txId'])
        
        # Filter edges to only those connecting nodes in our current set
        relevant_edges = input_edges[
            input_edges['txId1'].isin(node_ids) & 
            input_edges['txId2'].isin(node_ids)
        ]
        G = nx.from_pandas_edgelist(relevant_edges, 'txId1', 'txId2')
        degree_dict = dict(G.degree())
        target_df['node_degree'] = target_df['txId'].map(lambda x: degree_dict.get(x, 0))
        return target_df

In [12]:
train_df = df[(df['class'] == 0) & (df['time_step'] <= 10)].copy()
test_df = df[df['time_step'] > 10].copy()


## Feature Enrichment

In [13]:
train_df = add_degree_feature(train_df, edges)
test_df = add_degree_feature(test_df, edges)

In [14]:
#TODO: set embedding of node id

# MVP - abs scaler


train on licit class 0
training set: time-step 1-30
val: 31-49

In [15]:
## TODO: add embeddings for the connecting transactions. 

In [16]:
# TODO compare minmax scaler to maxabs scaler - how to test which one is better fitting

# Helper Function - train and get results

In [17]:
# Identify feature columns (excluding metadata)
metadata_cols = ['txId', 'time_step', 'class']

X_train_raw = train_df.drop(columns=metadata_cols)
X_test_raw = test_df.drop(columns=metadata_cols)


In [ ]:
def model_test_orchestrator(X_train, X_test, test_df, rank=16,  init='nndsvda'):
    

    nmf = NMF(n_components=rank, init=init, random_state=90, max_iter=500)
    W_train = nmf.fit_transform(X_train)
    H = nmf.components_     # The "Licit Patterns" matrix
    W_test = nmf.transform(X_test)

    X_test_recon = np.dot(W_test, H)

    # measure the error
    errors = np.mean((X_test - X_test_recon)**2, axis=1)
    
    test_results = pd.DataFrame({
        'txId': test_df['txId'].values,
        'time_step': test_df['time_step'].values,
        'class': test_df['class'].values,
        'anomaly_score': errors
    })
    
    # normalize error: 
    test_results['norm_error'] = MinMaxScaler().fit_transform(test_results[['anomaly_score']]).flatten()

    return nmf, H, W_test, test_results



In [19]:
def conf_matrix_maker(test_results, th=0.1):
    preds = test_results[test_results['class'] < 2].copy()

    tp = test_results[(test_results['class']==1) & (test_results['norm_error'] > th)]    # 2
    tn = test_results[(test_results['class']==0) & (test_results['norm_error'] < th)]    # 28,920
    fp = test_results[(test_results['class']==0) & (test_results['norm_error'] > th)]    # 1155
    fn = test_results[(test_results['class']==1) & (test_results['norm_error'] < th)]    # 4019

    # If the model says "fraud," is it actually fraud? - having few false alarm - high precision reduces fp. 
    precision = tp.shape[0] / (tp.shape[0] + fp.shape[0])

    # Did the model catch all the fraud, or did some slip through - how many actual positive were predicted as positives
    recall = tp.shape[0] / (fn.shape[0] + tp.shape[0])
    return precision, recall


## Test1: MaxAbsScaler

In [20]:
scaler = MaxAbsScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# add +1 to avoid negative values
X_train = X_train_scaled + 1.0
X_test = X_test_scaled + 1.0

In [21]:
rows, cols = np.where(X_test < 0)
print(list(zip(rows, cols)))

[(np.int64(1103), np.int64(100)), (np.int64(1103), np.int64(102)), (np.int64(9774), np.int64(100)), (np.int64(12858), np.int64(100)), (np.int64(17166), np.int64(100)), (np.int64(17166), np.int64(102)), (np.int64(22755), np.int64(100)), (np.int64(22755), np.int64(102)), (np.int64(45248), np.int64(100)), (np.int64(45248), np.int64(102)), (np.int64(61112), np.int64(100)), (np.int64(61112), np.int64(102)), (np.int64(72228), np.int64(100)), (np.int64(91136), np.int64(99)), (np.int64(97011), np.int64(99)), (np.int64(97011), np.int64(100)), (np.int64(97011), np.int64(102)), (np.int64(97884), np.int64(100)), (np.int64(98010), np.int64(100)), (np.int64(101520), np.int64(100)), (np.int64(130062), np.int64(100)), (np.int64(134160), np.int64(100))]


there are 22 negative values, stamming from columns 100, 102 and 99 - all aggregative. 

In [22]:
X_test = np.maximum(X_test, 1e-9)


In [ ]:
nmf, H, W_test, preds = model_test_orchestrator(X_train, X_test, test_df=test_df, rank=16)
preds


In [27]:
precision, recall = conf_matrix_maker(preds)
precision, recall

(0.0, 0.0)

## Test2 MinMax scaler + log

In [28]:
sample_size = 1000


In [40]:
# Identify feature columns (excluding metadata)
metadata_cols = ['txId', 'time_step', 'class']

sample_size = 1000

train_df_sample = train_df.sample(sample_size).copy()
test_df_sample = test_df.sample(sample_size).copy()
X_train_raw = train_df_sample.drop(columns=metadata_cols)
X_test_raw = test_df_sample.drop(columns=metadata_cols)

In [42]:
train_mins = X_train_raw.min()
test_mins = X_test_raw.min()
X_train_pos = (X_train_raw - train_mins) + 1.0
X_test_pos = (X_test_raw - test_mins) + 1.0
X_train_pos =  np.maximum(X_train_pos, 1e-12)
X_test_pos = np.maximum(X_test_pos, 1e-12)




In [43]:

# Step B: Log Transform (CRITICAL for power-law data like Bitcoin)
X_train_log = np.log1p(X_train_pos)
X_test_log = np.log1p(X_test_pos)


In [44]:

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train_log)
X_test = scaler.transform(X_test_log)

In [45]:
(X_train < 0).any().any()

np.False_

In [ ]:
#TODO change error

In [46]:
nmf, H, W_test, preds = model_test_orchestrator(X_train, X_test, test_df=test_df_sample, rank=16)
precision, recall = conf_matrix_maker(preds)
precision, recall

len errors 1000


(0.16666666666666666, 0.2)

In [ ]:
# preds = model_test_orchestrator(X_train, X_test, rank=None)
# precision, recall = conf_matrix_maker(preds)

In [47]:
type(X_train)

numpy.ndarray

In [50]:
ranks = [ 5, 10, 30, 50, 165]

results = {}

for k in ranks: 
    nmf, H, W_test, preds_df = model_test_orchestrator(X_train, X_test, test_df_sample, rank=k)
    precision, recall = conf_matrix_maker(preds_df)

    results[f'rank_{k}']: {"n_m_f": nmf, 'H_matrix': H, "w_test": W_test, "test_results": preds_df}
    print(f'rank {k}:\n precision: {precision}\n recall: {recall}')

len errors 1000
rank 5:
 precision: 0.09210526315789473
 recall: 0.23333333333333334
len errors 1000
rank 10:
 precision: 0.1
 recall: 0.2


/home/tiggi/Documents/IU_projects/bitcoin_rec/cc_fraud_NMF/.venv/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


len errors 1000
rank 30:
 precision: 0.0
 recall: 0.0


/home/tiggi/Documents/IU_projects/bitcoin_rec/cc_fraud_NMF/.venv/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


len errors 1000
rank 50:
 precision: 0.07142857142857142
 recall: 0.03333333333333333


/home/tiggi/Documents/IU_projects/bitcoin_rec/cc_fraud_NMF/.venv/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


len errors 1000
rank 165:
 precision: 0.0
 recall: 0.0


/home/tiggi/Documents/IU_projects/bitcoin_rec/cc_fraud_NMF/.venv/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1720: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
